# Group 2 - Fine-Tuning for Classification

A sentiment analysis notebook using PhoBERT and the **Vietnamese Students’ Feedback Corpus (UIT-VSFC)**.

In [1]:
!pip install torch transformers datasets evaluate accelerate pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.7 MB/s eta 0:00:00


In [2]:
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from pyvi import ViTokenizer
import numpy as np
import evaluate
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

## 1. Data Acquisition and Dataset Construction
This block handles the manual retrieval of raw data files from the official repository. Since the dataset is structured into separate files for sentences and labels, we perform a manual merge.

* **Logic:** Downloads text and label files for `train`, `validation`, and `test` splits.
* **Method:** Uses `pandas` to concatenate text strings with their corresponding integer labels and converts the resulting DataFrames into a Hugging Face `DatasetDict` object.

In [3]:
urls = {
    "train": {
        "sents": "https://drive.google.com/uc?id=1nzak5OkrheRV1ltOGCXkT671bmjODLhP&export=download",
        "labels": "https://drive.google.com/uc?id=1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv&export=download"
    },
    "validation": {
        "sents": "https://drive.google.com/uc?id=1sMJSR3oRfPc3fe1gK-V3W5F24tov_517&export=download",
        "labels": "https://drive.google.com/uc?id=1GiY1AOp41dLXIIkgES4422AuDwmbUseL&export=download"
    },
    "test": {
        "sents": "https://drive.google.com/uc?id=1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n&export=download",
        "labels": "https://drive.google.com/uc?id=1vkQS5gI0is4ACU58-AbWusnemw7KZNfO&export=download"
    }
}

def create_dataset(split):
    sents = pd.read_csv(urls[split]["sents"], sep="\t", header=None, names=["sentence"], on_bad_lines='skip', quoting=3)
    labels = pd.read_csv(urls[split]["labels"], header=None, names=["label"])

    df = pd.concat([sents, labels], axis=1).dropna()
    df["label"] = df["label"].astype(int)
    return Dataset.from_pandas(df)

dataset = DatasetDict({
    "train": create_dataset("train"),
    "validation": create_dataset("validation"),
    "test": create_dataset("test")
})

## 2. Vietnamese-Specific Preprocessing
Vietnamese text requires specialized processing due to its unique syllable structure. This block prepares the raw text for the Transformer model.

* **Word Segmentation:** Utilizes `pyvi` (ViTokenizer) to group syllables into proper words (e.g., "sinh viên" becomes "sinh_viên"). This is a prerequisite for PhoBERT.
* **Tokenization:** Employs `AutoTokenizer` to convert segmented text into input IDs and attention masks.
* **Configuration:** Applies a `max_length` of 128 with padding and truncation to ensure uniform tensor shapes within batches.

In [4]:
def tokenize_vi(example):
    example["sentence"] = ViTokenizer.tokenize(str(example["sentence"]))
    return example

dataset = dataset.map(tokenize_vi)

Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

Map:   0%|          | 0/3166 [00:00<?, ? examples/s]

In [5]:
MODEL_NAME = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

Map:   0%|          | 0/3166 [00:00<?, ? examples/s]

## 3. Model Configuration and Fine-Tuning
This block defines the model architecture and the training loop parameters.

* **Architecture:** Loads `AutoModelForSequenceClassification` with 3 target labels. This automatically attaches a Linear Classifier Head on top of the pre-trained RoBERTa encoder.
* **Optimization:** Uses the `Trainer` API to handle the training loop. Key parameters include:
    * `learning_rate`: Set to 2e-5 for stable fine-tuning.
    * `eval_strategy`: Evaluates model performance at the end of every epoch.
    * `processing_class`: Manages the data transformation during the training process.
* **Loss Function:** The model defaults to Cross-Entropy Loss to optimize the classifier head weights.

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

In [8]:
training_args = TrainingArguments(
    output_dir="./phobert_vsfc_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.291732,0.247251,0.947568
2,0.194538,0.236329,0.948831
3,0.130199,0.243371,0.947568
4,0.107525,0.264199,0.948200
5,0.075668,0.273523,0.947568
6,0.064465,0.298645,0.950726
7,0.053505,0.314399,0.945673
8,0.042166,0.319939,0.950095
9,0.033247,0.363313,0.946936
10,0.018684,0.397632,0.943778


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=10725, training_loss=0.06597049179610673, metrics={'train_runtime': 4449.0292, 'train_samples_per_second': 38.523, 'train_steps_per_second': 2.411, 'total_flos': 1.127375216610048e+16, 'train_loss': 0.06597049179610673, 'epoch': 15.0})

## 4. Inference and Sentiment Prediction
The final block encapsulates the trained model into a usable function for real-world text inputs.

* **Workflow:**
    1. Input raw Vietnamese string.
    2. Apply `ViTokenizer` for segmentation.
    3. Tokenize and move tensors to the available device (CPU/GPU).
    4. Execute a forward pass without gradient calculation.
    5. Map the output `logits` to human-readable labels: **Negative (0)**, **Neutral (1)**, or **Positive (2)**.

In [9]:
print("--- Kết quả trên tập Test ---")
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

--- Kết quả trên tập Test ---


{'eval_loss': 0.2825959026813507, 'eval_accuracy': 0.936197094125079, 'eval_runtime': 22.334, 'eval_samples_per_second': 141.757, 'eval_steps_per_second': 17.731, 'epoch': 15.0}


In [10]:
def predict_sentiment(text):
    text_segmented = ViTokenizer.tokenize(text)
    inputs = tokenizer(text_segmented, return_tensors="pt", truncation=True, padding=True).to("cuda" if torch.cuda.is_available() else "cpu")
    model.to(inputs['input_ids'].device)

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class = torch.argmax(logits, dim=1).item()
    labels_map = {0: "Tiêu cực (Negative)", 1: "Trung lập (Neutral)", 2: "Tích cực (Positive)"}

    return labels_map[predicted_class]

In [11]:
print(predict_sentiment("Giảng viên này hay và dễ hiểu nhưng cho bài tập khó quá."))

Tiêu cực (Negative)
